In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from scipy.stats import chi2

## Hawke's log-likelihood

In [ ]:
def log_likelihood(params, t, T):
    mu, alpha, beta = params

    if mu <= 0 or alpha <= 0 or beta <= 0 or alpha >= beta:
        return 1e10
    
    n=len(t)
    log_lambda_sum = 0
    for i in range(n):
        excitation = np.sum(alpha * np.exp(-beta * (t[i]-t[:i])))
        log_lambda_sum += np.log(mu + excitation)
    
    integral = mu * T + np.sum((alpha / beta) * (1 - np.exp(-beta * (T - t))))
    return (-log_lambda_sum + integral)

## Pure poission process for Likelihood Ratio Test

In [ ]:
def pure_poisson_log_likelihood(mu, t, T):
    n = len(t)
    return -(n * np.log(mu) - mu * T)

## Compute Maximum Likelihood: alpha, beta, gamma

In [ ]:
df = pd.read_excel('CDX_HY_Defaults.xlsx')
dates = pd.to_datetime(df['Default_Date']).sort_values().reset_index(drop=True)

start_date = dates.min()
end_date = dates.max()

t = (dates - start_date).dt.days.values
T = (end_date - start_date).days

initial_guess = [0.005, 0.01, 0.1] # = [mu, alpha, beta]
result = minimize(log_likelihood, initial_guess, args=(t,T), method='L-BFGS-B')
mu, alpha, beta = result.x

print(f"Estimaed mu: {mu:.4f}")
print(f"Estimated alpha: {alpha:.4f}")
print(f"Estimated beta: {beta:.4f}")

## Likelihood Ratio Test (LRT)

In [ ]:
result_poisson = minimize(pure_poisson_log_likelihood, [0.01], args=(t, T), method='L-BFGS-B', bounds=[(1e-6, None)])
ll_null = -result_poisson.fun
ll_alt = -result.fun

D = 2 * (ll_alt - ll_null)
p_val = chi2.sf(D, df=2)

is_significant = p_val < 0.05

print(f"Likelihood Ratio: {D:.4}")
print(f"P-value: {p_val:.6f}")
print(f"Is Significant: {is_significant}")

In [ ]:
def compute_intensity(t_grid, t_events, mu, alpha, beta):
    """Beräknar lambda(t) för varje punkt i ett tidsrutnät."""
    lambdas = np.full_like(t_grid, mu)
    for i, t_val in enumerate(t_grid):
        past_events = t_events[t_events < t_val]
        if len(past_events) > 0:
            excitation = np.sum(alpha * np.exp(-beta * (t_val - past_events)))
            lambdas[i] += excitation
    return lambdas

t_grid = np.linspace(0, T, num=int(T))
lambdas = compute_intensity(t_grid, t, mu, alpha, beta)

plt.figure(figsize=(12, 6))
plt.plot(t_grid, lambdas, label='Hawkes Intensitet $\lambda(t)$', color='firebrick', lw=1.5)
plt.axhline(y=mu, color='black', linestyle='--', alpha=0.5, label='Basrisk $\mu$ (Poisson)')

plt.vlines(t, ymin=0, ymax=lambdas.max()*0.05, color='blue', alpha=0.6, label='Faktiska Defaults')

plt.title(f'Kreditriskens dynamik i CDX HY (alpha={alpha:.4f}, beta={beta:.4f})')
plt.xlabel('Days since start date (S11)')
plt.ylabel('Sannolikhet för default per tidsenhet')
plt.legend()
plt.grid(alpha=0.3)
plt.show()